# Full-history 13F orientation

This notebook explains how the local 13F product is physically organized and what one observation means. It measures table shape, schema stability, economic reporting dates, filing dates, availability dates, filing cadence, amendments, and a bounded sample of holding fields.

It is an audit and learning notebook, not a cleaning pipeline, embedding model, or backtest.

## Questions answered

1. What tables are included, and how large are they?
2. What does one row in the holdings table represent?
3. Which variables are available?
4. What is the difference between the portfolio date, publication time, and research availability time?
5. Is the data economically quarterly even though the files are partitioned monthly?
6. How common are amendments and multiple filings for one manager-quarter?
7. Which decisions remain before an investor-by-asset matrix can be built?

The notebook does not print cloud URIs, raw manager names, raw issuer names, CUSIPs, tickers, or individual holdings. Executed outputs stay local and must not be committed.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.full_history_13f import (
    REQUIRED_FILING_COLUMNS,
    bounded_parquet_sample,
    filing_lag_summary,
    manager_period_multiplicity,
    parquet_schema,
    parquet_table_inventory,
    prepare_13f_filings,
    quarterly_filing_summary,
    read_parquet_columns,
    timing_field_summary,
)
from src.holdings_exploration import (
    DATASET_SPECS,
    category_profile,
    identifier_profile,
    schema_profile,
    value_profile,
)

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

## 1. Locate the ignored local product

The synchronized data live below `data/full_history/`, which is ignored by Git. The manifest is intentionally not displayed because it contains private operational metadata.

In [ ]:
PRODUCT_ROOT = PROJECT_ROOT / "data" / "full_history" / "13f" / "v1"
TABLES_ROOT = PRODUCT_ROOT / "tables"
HOLDINGS_DIR = TABLES_ROOT / "holdings"
FILINGS_DIR = TABLES_ROOT / "filings"
DICTIONARY_DIR = TABLES_ROOT / "variable_dictionary"

required_paths = [PRODUCT_ROOT, HOLDINGS_DIR, FILINGS_DIR, DICTIONARY_DIR]
missing_paths = [str(path.relative_to(PROJECT_ROOT)) for path in required_paths if not path.is_dir()]
if missing_paths:
    raise FileNotFoundError(
        "The local 13F mirror is incomplete. Missing: " + ", ".join(missing_paths)
    )

pd.DataFrame(
    [{"local_component": path.name, "available": path.is_dir()} for path in required_paths]
)

## 2. Physical organization, shape, and size

Parquet footers contain row counts and schemas. Reading those footers gives a complete inventory without loading the holdings payload into memory. `schema_variants > 1` means at least one partition has a different column type or column set.

In [ ]:
inventory = parquet_table_inventory(PRODUCT_ROOT)
inventory_display = inventory.copy()
inventory_display["storage_gib"] = inventory_display["storage_gib"].round(3)
display(inventory_display)

The main modeling input is the long-format `holdings` table:

$$\text{one row} = \text{one manager} \times \text{one filing} \times \text{one reported holding line}.$$

It is not yet an investor-by-asset matrix. A matrix requires amendment resolution, asset mapping, aggregation, eligibility filters, and a point-in-time decision rule.

In [ ]:
holdings_inventory = inventory.loc[
    inventory["table"].eq("holdings"),
    ["table", "parquet_files", "rows", "columns_min", "columns_max", "schema_variants", "storage_gib"],
]
holdings_inventory = holdings_inventory.assign(
    storage_gib=holdings_inventory["storage_gib"].round(3)
)
display(holdings_inventory)

## 3. Holdings variables

The schema below comes directly from every holdings partition. The variable dictionary adds the product's declared role, unit, date semantics, missing-state policy, and leakage warning.

In [ ]:
holdings_schema = parquet_schema(HOLDINGS_DIR)
dictionary = read_parquet_columns(
    DICTIONARY_DIR,
    [
        "table_name",
        "column_name",
        "role",
        "unit",
        "definition",
        "date_semantics",
        "null_policy",
        "leakage_risk",
    ],
)
holdings_dictionary = dictionary.loc[
    dictionary["table_name"].eq("holdings"),
    ["column_name", "role", "unit", "definition", "date_semantics", "null_policy", "leakage_risk"],
]
holdings_variables = holdings_schema.merge(
    holdings_dictionary,
    how="left",
    left_on="column",
    right_on="column_name",
).drop(columns="column_name")
display(holdings_variables)

## 4. Three different times

- **Economic date — `period_of_report`:** when the portfolio state applies, normally a calendar quarter-end.
- **Published time — `filed_at`:** when the manager filed the form.
- **Available time — `availability_timestamp_utc`:** the earliest timestamp at which our research is allowed to use it.

The filing lag is:

$$\text{filing lag} = \text{filed at} - \text{portfolio report date}.$$

A backtest must never use a quarter-end holding before its availability timestamp.

In [ ]:
filings_raw = read_parquet_columns(FILINGS_DIR, REQUIRED_FILING_COLUMNS)
filings = prepare_13f_filings(filings_raw)

display(timing_field_summary(filings))
display(
    pd.DataFrame(
        [
            {
                "filings": len(filings),
                "managers": filings["manager_cik"].nunique(dropna=True),
                "standard_quarter_end_rows": int(filings["is_standard_quarter_end"].sum()),
                "report_after_filing_violations": int((~filings["report_not_after_filing"]).sum()),
                "available_before_filing_violations": int((~filings["available_not_before_filing"]).sum()),
            }
        ]
    )
)

## 5. Recording-to-publication lag

The following table separates form types because a holdings report, a notice, and an amendment are economically different. Extreme minimum or maximum lags should be treated as audit observations, not silently winsorized.

In [ ]:
display(filing_lag_summary(filings))

availability_delay = filings["availability_delay_minutes"].dropna()
display(
    pd.DataFrame(
        [
            {
                "availability_delay_min_minutes": availability_delay.min(),
                "availability_delay_median_minutes": availability_delay.median(),
                "availability_delay_p90_minutes": availability_delay.quantile(0.90),
                "availability_delay_max_minutes": availability_delay.max(),
            }
        ]
    )
)

## 6. Quarterly economic cadence versus monthly file partitions

The local files are grouped by filing month because managers submit forms on different days. The underlying portfolio snapshots are normally synchronized calendar quarter-ends. The table therefore groups on the economic report date, not the file name or filing month.

In [ ]:
first_published_at = filings["filed_at_utc"].min()
inferred_product_start = (first_published_at - pd.offsets.QuarterEnd()).normalize()
core_mask = filings["is_standard_quarter_end"] & filings["report_date_utc"].ge(inferred_product_start)
core_filings = filings.loc[core_mask].copy()
quarterly = quarterly_filing_summary(core_filings)
quarterly["report_quarter"] = (
    quarterly["report_date_utc"].dt.tz_localize(None).dt.to_period("Q").astype("string")
)

display(pd.concat([quarterly.head(8), quarterly.tail(8)], ignore_index=True))

quarterly_by_year = (
    quarterly.assign(report_year=quarterly["report_date_utc"].dt.year)
    .groupby("report_year")
    .agg(
        quarter_snapshots=("report_date_utc", "nunique"),
        filings=("filings", "sum"),
        managers_median=("managers", "median"),
        holding_lines_reported=("holding_lines_reported", "sum"),
        amendments=("amendments", "sum"),
    )
    .reset_index()
)
display(quarterly_by_year)

Quarter-end report dates earlier than the inferred product window, as well as non-quarter-end dates, can arise from late filings, corrections, backfilled records, or malformed source values. They are preserved and summarized separately. Their presence does not mean the product has continuous coverage back to the earliest exceptional date.

In [ ]:
outside_core = filings.loc[~core_mask].copy()
display(
    pd.DataFrame(
        [
            {
                "inferred_continuous_start": inferred_product_start,
                "outside_core_filings": len(outside_core),
                "distinct_outside_core_report_dates": outside_core["report_date_utc"].nunique(dropna=True),
                "earliest_outside_core_report_date": outside_core["report_date_utc"].min(),
                "latest_outside_core_report_date": outside_core["report_date_utc"].max(),
                "missing_report_dates": int(outside_core["report_date_utc"].isna().sum()),
            }
        ]
    )
)

## 7. Filing forms, amendments, and manager-quarter multiplicity

`13F-HR` and `13F-HR/A` are holdings reports and amendments. Notice forms can have no information table. Multiple accessions for one manager and report date are not automatically duplicates: they may represent amendments or related filing states. This notebook counts them but does not choose which version to keep.

In [ ]:
form_summary = (
    filings.groupby(["form_type", "is_amendment"], dropna=False)
    .agg(
        filings=("accession_no", "nunique"),
        managers=("manager_cik", "nunique"),
        holding_lines_reported=("holdings_count_numeric", "sum"),
    )
    .reset_index()
)
display(form_summary)
display(manager_period_multiplicity(filings))

## 8. Portfolio-size orientation from the filing index

`holdings_count` is the number of reported lines attached to a filing. It is not yet the number of distinct eligible companies because one company may have multiple securities or reported lines.

In [ ]:
portfolio_shape = (
    filings.groupby("form_type", dropna=False)["holdings_count_numeric"]
    .agg(
        filings="size",
        minimum="min",
        p10=lambda values: values.quantile(0.10),
        median="median",
        p90=lambda values: values.quantile(0.90),
        p99=lambda values: values.quantile(0.99),
        maximum="max",
    )
    .reset_index()
)
display(portfolio_shape)

## 9. Bounded holdings-content sample

The complete holdings table is too large for an in-memory exploratory read. The next cell takes the first batch from a small number of evenly spaced filing-month partitions. This is suitable for learning field states and identifier coverage, but not for estimating population statistics.

Only aggregate profiles are displayed; individual holdings are not printed.

In [ ]:
sample_columns = [
    "holding_id",
    "accession_no",
    "manager_cik",
    "form_type",
    "period_of_report",
    "filed_at",
    "availability_timestamp_utc",
    "cusip",
    "ticker",
    "reported_issuer_cik",
    "resolved_issuer_cik",
    "issuer_provider_entity_id",
    "issuer_identity_confidence",
    "value_usd",
    "shares_or_principal_amount",
    "shares_or_principal_type",
    "put_call",
    "investment_discretion",
]
holdings_sample, sample_design = bounded_parquet_sample(
    HOLDINGS_DIR,
    sample_columns,
    max_files=7,
    rows_per_file=20_000,
)
display(sample_design)

In [ ]:
display(schema_profile(holdings_sample))
display(identifier_profile(holdings_sample, DATASET_SPECS["13f"]))
display(value_profile(holdings_sample, DATASET_SPECS["13f"]))
display(
    category_profile(
        holdings_sample,
        [
            "form_type",
            "put_call",
            "shares_or_principal_type",
            "investment_discretion",
            "issuer_identity_confidence",
        ],
    )
)

## 10. What we now know, and what remains undecided

This notebook establishes the physical tables, row grain, schema, timing fields, quarterly cadence, publication lag, form composition, amendment prevalence, and sampled holding states.

It deliberately does **not** decide:

1. whether an amendment replaces or supplements an earlier filing;
2. how to treat notice filings and confidential-treatment effects;
3. whether the model asset is a security, issuer, or company;
4. which position types qualify as long equity;
5. how manager reorganizations are handled through time;
6. the final coverage window and partial-quarter rule;
7. the standardized decision date after filings become available.

Those decisions belong in the next point-in-time data-spine milestone and must be supported by full-history evidence rather than a bounded sample.